# SepsisGuard — GRPO Training Smoke Test

Minimal notebook to verify the full training pipeline works end-to-end.
- Task 1 only, 10 training steps, 2 generations
- Records score **before** and **after** training
- Expected runtime: ~5-10 minutes on T4

In [ ]:
!pip install -q -U "unsloth[colab-new]" openenv-core "trl>=0.12" vllm datasets
!pip install -q requests httpx

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
%cd /content/sepsisguard
!pip install -q -e .
!nohup uvicorn server.app:app --host 0.0.0.0 --port 7860 > /content/uvicorn.log 2>&1 &

In [ ]:
import os, requests, json, re, uuid

# If notebook and server run on the SAME machine, use localhost.
# For Colab -> your laptop server, localhost will not work; use a public tunnel URL instead.
ENV_URL = os.environ.get("ENV_BASE_URL", "http://127.0.0.1:7860")
TASK = "task1_textbook"

class EnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def reset(self, task_name, seed, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/reset",
                          json={"task_name": task_name, "seed": seed},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def step(self, actions, session_id=None):
        headers = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/step",
                          json={"actions": actions},
                          headers=headers, timeout=30)
        r.raise_for_status()
        return r.json()

    def create_session(self):
        r = requests.post(f"{self.base_url}/session", timeout=10)
        r.raise_for_status()
        return r.json()["session_id"]

    def delete_session(self, session_id):
        try:
            requests.delete(f"{self.base_url}/session/{session_id}", timeout=5)
        except Exception:
            pass

env = EnvClient(ENV_URL)
info = env.reset(task_name=TASK, seed=42)
print(f"Connected to {ENV_URL}")
print(f"Tick: {info['info']['tick']}, Roles: {list(info['observations'].keys())}")

In [ ]:
# --- Before-training baseline score ---
import torch
from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician
from training.prompts import build_role_prompt

nurse_h, lab_h, pharma_h, phys_h = HeuristicNurse(), HeuristicLab(), HeuristicPharmacist(), HeuristicPhysician()
heuristic_agents = {"nurse": nurse_h, "lab": lab_h, "pharmacist": pharma_h, "physician": phys_h}

def run_heuristic_episode(env_client, task, seed):
    sid = env_client.create_session()
    bundle = env_client.reset(task_name=task, seed=seed, session_id=sid)
    done = False
    while not done:
        obs = bundle["observations"]
        actions = {r: heuristic_agents[r].decide(obs[r]) for r in heuristic_agents}
        bundle = env_client.step(actions, session_id=sid)
        done = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": sid}, timeout=30).json()
    env_client.delete_session(sid)
    return grader.get("score", 0.0)

def make_llm_agent_fn(mdl, tok, target_role):
    def agent_fn(role, obs):
        if role != target_role:
            return heuristic_agents[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tok(prompt, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=128, do_sample=False)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and "operation" in parsed:
                return parsed
        except Exception:
            pass
        m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
        return heuristic_agents[role].decide(obs)
    return agent_fn

def run_llm_episode(env_client, task, seed, agent_fn):
    sid = env_client.create_session()
    bundle = env_client.reset(task_name=task, seed=seed, session_id=sid)
    done = False
    while not done:
        obs = bundle["observations"]
        actions = {r: agent_fn(r, obs[r]) for r in ("nurse", "lab", "pharmacist", "physician")}
        bundle = env_client.step(actions, session_id=sid)
        done = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": sid}, timeout=30).json()
    env_client.delete_session(sid)
    return grader.get("score", 0.0)

# Heuristic baseline
baseline_scores = [run_heuristic_episode(env, TASK, 100 + i) for i in range(3)]
print(f"Heuristic baseline: {[round(s, 4) for s in baseline_scores]}  mean={sum(baseline_scores)/len(baseline_scores):.4f}")

# Pre-training LLM score (nurse role)
FastLanguageModel.for_inference(model)
pre_fn = make_llm_agent_fn(model, tokenizer, "nurse")
pre_scores = [run_llm_episode(env, TASK, 100 + i, pre_fn) for i in range(3)]
print(f"Pre-training (nurse): {[round(s, 4) for s in pre_scores]}  mean={sum(pre_scores)/len(pre_scores):.4f}")

In [ ]:
# --- Before-training baseline score + detailed step tracing ---
import os
import sys
import json
import re
import importlib
import inspect
import requests
import torch
import time
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from training.prompts import build_role_prompt

# Runtime knobs: switch FAST_MODE to False for full verbose evaluation.
FAST_MODE = True
HEURISTIC_EPISODES = 2 if FAST_MODE else 3
LLM_EPISODES = 2 if FAST_MODE else 3
MAX_NEW_TOKENS = 64 if FAST_MODE else 128
SHOW_STEP_LOGS = True
LOG_EVERY_N_STEPS = 1

# 1) Path configuration
PROJECT_ROOT = "/content/sepsisguard"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
importlib.invalidate_caches()

# 2) Robust class resolution (handles branch/version naming differences)
def resolve_agent_class(module_name: str, preferred_name: str, role_hint: str):
    mod = importlib.import_module(module_name)
    if hasattr(mod, preferred_name):
        return getattr(mod, preferred_name)

    candidates = []
    for name, obj in vars(mod).items():
        if not inspect.isclass(obj):
            continue
        if obj.__module__ != mod.__name__:
            continue
        if not hasattr(obj, "decide"):
            continue
        score = 0
        lname = name.lower()
        if role_hint in lname:
            score += 2
        if "heuristic" in lname:
            score += 2
        if "agent" in lname:
            score += 1
        candidates.append((score, name, obj))

    if not candidates:
        exported = [k for k in vars(mod).keys() if not k.startswith("_")]
        raise ImportError(f"No usable agent class found in {module_name}. Exported symbols: {exported}")

    candidates.sort(key=lambda x: (-x[0], x[1]))
    chosen_name = candidates[0][1]
    print(f"Resolved {module_name}: using {chosen_name} (preferred {preferred_name} not found)")
    return candidates[0][2]

NurseCls = resolve_agent_class("agents.nurse", "HeuristicNurse", "nurse")
LabCls = resolve_agent_class("agents.lab", "HeuristicLab", "lab")
PharmaCls = resolve_agent_class("agents.pharmacist", "HeuristicPharmacist", "pharmac")
PhysCls = resolve_agent_class("agents.physician", "HeuristicPhysician", "physician")

heuristic_agents = {
    "nurse": NurseCls(),
    "lab": LabCls(),
    "pharmacist": PharmaCls(),
    "physician": PhysCls(),
}
print("System: Heuristic agents ready.")

# 3) Episode runner with per-step tracing
def run_episode(env_client, task, seed, agent_fn, label, show_steps=SHOW_STEP_LOGS):
    sid = env_client.create_session()
    bundle = env_client.reset(task_name=task, seed=seed, session_id=sid)
    done = False
    step_idx = 0
    step_logs = []
    t0 = time.perf_counter()

    try:
        while not done:
            obs = bundle["observations"]
            actions = {r: agent_fn(r, obs[r]) for r in ("nurse", "lab", "pharmacist", "physician")}
            bundle = env_client.step(actions, session_id=sid)
            done = bundle["done"]
            step_idx += 1

            tick = bundle.get("info", {}).get("tick", step_idx)
            rewards = bundle.get("rewards", {})

            step_logs.append({
                "step": step_idx,
                "tick": tick,
                "actions": actions,
                "rewards": rewards,
            })

            if show_steps and (step_idx % LOG_EVERY_N_STEPS == 0 or done):
                ops = {k: v.get("operation", "<none>") for k, v in actions.items()}
                print(f"[{label}] step={step_idx:02d} tick={tick} ops={ops} rewards={rewards}")

        grader = requests.get(
            f"{env_client.base_url}/grader",
            headers={"X-Session-Id": sid},
            timeout=30,
        ).json()
        score = float(grader.get("score", 0.0))
    finally:
        env_client.delete_session(sid)

    elapsed = time.perf_counter() - t0
    return score, step_logs, elapsed

# 4) Heuristic baseline (3 episodes)
print("\n[1/2] Heuristic baseline with per-step logs")
baseline_scores = []
heur_times = []
for i in tqdm(range(HEURISTIC_EPISODES), desc="Heuristic episodes"):
    def h_fn(role, role_obs):
        return heuristic_agents[role].decide(role_obs)
    score, _, elapsed = run_episode(env, TASK, 100 + i, h_fn, label=f"HEUR-{i+1}")
    baseline_scores.append(score)
    heur_times.append(elapsed)
    print(f"[HEUR-{i+1}] episode_score={score:.4f} time={elapsed:.1f}s")

# 5) LLM pre-training eval (nurse-only LLM) + parse accuracy
print("\n[2/2] Pre-training LLM (nurse role) with per-step logs")
FastLanguageModel.for_inference(model)

parse_stats = {"attempts": 0, "valid_action": 0, "fallbacks": 0}

def nurse_llm_fn(role, role_obs):
    if role != "nurse":
        return heuristic_agents[role].decide(role_obs)

    parse_stats["attempts"] += 1
    prompt = build_role_prompt(role_obs, role)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "operation" in parsed:
            parse_stats["valid_action"] += 1
            return parsed
    except Exception:
        pass

    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
    if m:
        try:
            parsed = json.loads(m.group(0))
            if isinstance(parsed, dict) and "operation" in parsed:
                parse_stats["valid_action"] += 1
                return parsed
        except Exception:
            pass

    parse_stats["fallbacks"] += 1
    return heuristic_agents[role].decide(role_obs)

pre_scores = []
llm_times = []
for i in tqdm(range(LLM_EPISODES), desc="Pre-train LLM episodes"):
    score, _, elapsed = run_episode(env, TASK, 100 + i, nurse_llm_fn, label=f"LLM-{i+1}")
    pre_scores.append(score)
    llm_times.append(elapsed)
    print(f"[LLM-{i+1}] episode_score={score:.4f} time={elapsed:.1f}s")

# 6) Final report
heur_mean = sum(baseline_scores) / len(baseline_scores)
pre_mean = sum(pre_scores) / len(pre_scores)
parse_accuracy = (parse_stats["valid_action"] / max(1, parse_stats["attempts"]))

print("\n" + "=" * 56)
print("BEFORE-TRAINING EVALUATION SUMMARY")
print("=" * 56)
print(f"Heuristic mean score:   {heur_mean:.4f}  {baseline_scores}")
print(f"Heuristic avg time:     {sum(heur_times)/len(heur_times):.1f}s")
print(f"LLM pre-train mean:     {pre_mean:.4f}  {pre_scores}")
print(f"LLM avg time:           {sum(llm_times)/len(llm_times):.1f}s")
print(f"LLM parse accuracy:     {parse_accuracy:.2%} ({parse_stats['valid_action']}/{parse_stats['attempts']})")
print(f"LLM fallback count:     {parse_stats['fallbacks']}")
print("=" * 56)

In [ ]:
# --- Minimal GRPO training (with progress, no rollout-generation stall) ---
import time
from tqdm.auto import tqdm
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from training.reward_shaping import make_online_sepsis_reward_fn, format_reward_fn
from training.prompts import build_role_prompt
from transformers import TrainerCallback
import torch

ROLES = ("nurse", "lab", "pharmacist", "physician")

# 1) Fast prompt-only rollout collection with visible step progress.
# This avoids the long stall from collect_rollouts(), which generates many LLM completions.
if "rollouts" not in globals() or not rollouts:
    print("No rollouts found. Collecting prompt-only rollouts (fast mode)...")
    rollout_episodes = 2 if "FAST_MODE" not in globals() or FAST_MODE else 1
    max_steps_per_episode = 48  # task1 upper bound, acts as a safety cap
    rollouts = []

    for ep in tqdm(range(rollout_episodes), desc="Collecting episodes"):
        sid = env.create_session()
        bundle = env.reset(task_name=TASK, seed=500 + ep, session_id=sid)
        done = False
        step_idx = 0

        with tqdm(total=max_steps_per_episode, desc=f"Episode {ep+1} steps", leave=False) as step_pbar:
            while not done and step_idx < max_steps_per_episode:
                obs = bundle["observations"]

                # Collect prompts for all roles at current state.
                for role in ROLES:
                    rollouts.append({
                        "prompt": build_role_prompt(obs[role], role),
                        "role": role,
                    })

                # Use heuristic actions for fast environment stepping.
                actions = {role: heuristic_agents[role].decide(obs[role]) for role in ROLES}
                bundle = env.step(actions, session_id=sid)
                done = bundle["done"]
                step_idx += 1
                step_pbar.update(1)

        env.delete_session(sid)
        print(f"[episode {ep+1}/{rollout_episodes}] steps={step_idx} total_prompts={len(rollouts)}")
else:
    print(f"Using existing rollouts: {len(rollouts)} items")

# 2) Build dataset
train_dataset = Dataset.from_list([{"prompt": r["prompt"]} for r in rollouts])
print(f"Train dataset size: {len(train_dataset)}")

# 3) Configure trainer (progress-friendly logging + safe precision selection)
FastLanguageModel.for_training(model)

has_cuda = torch.cuda.is_available()
if has_cuda:
    major, _ = torch.cuda.get_device_capability()
    use_bf16 = major >= 8  # Ampere or newer
    use_fp16 = not use_bf16
    print(f"CUDA device: {torch.cuda.get_device_name(0)} | bf16={use_bf16} fp16={use_fp16}")
else:
    use_bf16 = False
    use_fp16 = False
    print("CUDA not available. Using fp32 precision.")

cfg = GRPOConfig(
    output_dir="./sepsis-grpo-test",
    num_generations=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=10,
    learning_rate=2e-5,
    warmup_steps=2,
    logging_steps=1,
    save_steps=10,
    max_prompt_length=3000,
    max_completion_length=128,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    disable_tqdm=False,
 )

reward_fn_env_obj = make_online_sepsis_reward_fn(
    env_url=ENV_URL, task_name=TASK, seed=42, max_steps_per_eval=3,
 )

# Unsloth GRPOTrainer expects reward callables with a __name__ attribute.
def reward_fn_env(*args, **kwargs):
    return reward_fn_env_obj(*args, **kwargs)

reward_fn_env.__name__ = "reward_fn_env"

# Extra explicit Step-4 progress printer (in addition to trainer/tqdm).
class StepProgressCallback(TrainerCallback):
    def __init__(self, total_steps: int):
        self.total_steps = max(1, int(total_steps))

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = int(state.global_step)
        pct = 100.0 * step / self.total_steps
        bits = [f"step {step}/{self.total_steps} ({pct:.1f}%)"]
        for key in ("loss", "reward", "mean_reward", "grad_norm", "learning_rate"):
            if key in logs:
                val = logs[key]
                if isinstance(val, float):
                    bits.append(f"{key}={val:.4g}")
                else:
                    bits.append(f"{key}={val}")
        print("[train] " + " | ".join(bits))

progress_cb = StepProgressCallback(cfg.max_steps)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn_env, format_reward_fn],
    args=cfg,
    train_dataset=train_dataset,
    callbacks=[progress_cb],
 )

# 4) Train
print("Starting 10-step smoke test training...")
t0 = time.perf_counter()
trainer.train()
train_elapsed = time.perf_counter() - t0
print(f"Training complete (smoke test passed) in {train_elapsed:.1f}s.")

In [ ]:
# --- Post-training score (with progress) ---
import time
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

# Reuse episode count from Cell 6 if available.
post_eval_episodes = LLM_EPISODES if "LLM_EPISODES" in globals() else 2
post_scores = []
post_times = []

for i in tqdm(range(post_eval_episodes), desc="Post-train LLM episodes"):
    score, _, elapsed = run_episode(env, TASK, 200 + i, nurse_llm_fn, label=f"POST-{i+1}")
    post_scores.append(score)
    post_times.append(elapsed)
    print(f"[POST-{i+1}] episode_score={score:.4f} time={elapsed:.1f}s")

print("=" * 56)
print("SMOKE TEST RESULTS")
print("=" * 56)
base_mean = sum(baseline_scores) / len(baseline_scores) if "baseline_scores" in globals() else float("nan")
pre_mean_local = sum(pre_scores) / len(pre_scores) if "pre_scores" in globals() else float("nan")
post_mean = sum(post_scores) / len(post_scores)
print(f"Heuristic baseline:  {base_mean:.4f}  {[round(s,4) for s in baseline_scores] if 'baseline_scores' in globals() else 'N/A'}")
print(f"Pre-training (LLM):  {pre_mean_local:.4f}  {[round(s,4) for s in pre_scores] if 'pre_scores' in globals() else 'N/A'}")
print(f"Post-training (LLM): {post_mean:.4f}  {[round(s,4) for s in post_scores]}")
print(f"Post avg time:        {sum(post_times)/len(post_times):.1f}s")
if "pre_scores" in globals():
    print(f"Delta (post - pre):  {post_mean - pre_mean_local:+.4f}")
if "baseline_scores" in globals():
    print(f"Delta (post - base): {post_mean - base_mean:+.4f}")
print()
if "pre_scores" in globals() and post_mean > pre_mean_local:
    print("Pipeline works: score improved after training.")
elif "pre_scores" in globals() and post_mean == pre_mean_local:
    print("Pipeline ran but score unchanged (10 steps may not be enough to see improvement).")
elif "pre_scores" in globals():
    print("Score decreased slightly -- normal for only 10 steps. Pipeline is functional.")
else:
    print("Post-training evaluation ran. Run Cell 6 first for full before/after deltas.")
print()
print("If this cell ran without errors, the full training pipeline is verified.")
print("Proceed to colab_training.ipynb for the real 200-step run.")